# Phân tích Recall theo kích thước đối tượng

## Workflow

1. Khai báo đường dẫn đến `best.pt`, ảnh validation và nhãn validation.
2. Lấy danh sách ảnh validation và kiểm tra dữ liệu.
3. Đọc nhãn thật: đổi box YOLO sang tọa độ pixel và chia thành `small`, `medium`, `large`.
4. Load model baseline và predict toàn bộ ảnh validation với `imgsz=640`.
5. Với từng ảnh, ghép prediction với ground truth khi **cùng class** và **IoU >= 0.5**.
6. Đếm số ground truth, số phát hiện đúng và số bỏ sót theo từng kích thước.
7. Tính `Recall = Detected / Ground truth`.

Nếu Recall của `small` thấp hơn rõ rệt so với `medium` và `large`, baseline có dấu hiệu gặp khó khăn với vật thể nhỏ.

## Bước 1 — Cài đặt và import thư viện

In [ ]:
%pip install -q ultralytics

from pathlib import Path
import numpy as np
from ultralytics import YOLO

## Bước 2 — Khai báo đường dẫn và thông số

Các đường dẫn dưới đây là minh họa. Hãy thay bằng đường dẫn thật trên Colab.

In [ ]:
# Đường dẫn đến weights baseline
BEST_PT = Path("/content/DACS/runs/detect/train/weights/best.pt")

# Thư mục ảnh và nhãn validation
VAL_IMAGES = Path("/content/DACS/datasets/VisDrone/images/val")
VAL_LABELS = Path("/content/DACS/datasets/VisDrone/labels/val")

# Giữ nguyên khi so sánh baseline với model cải tiến
IMGSZ = 640
CONFIDENCE = 0.25
IOU_THRESHOLD = 0.50

## Bước 3 — Lấy danh sách ảnh validation

`glob("*.jpg")` lấy tất cả ảnh JPG. Mỗi ảnh `abc.jpg` sẽ dùng nhãn thật `abc.txt`.

In [ ]:
image_files = sorted(VAL_IMAGES.glob("*.jpg"))
label_files = sorted(VAL_LABELS.glob("*.txt"))

print("Weights tồn tại:", BEST_PT.exists())
print("Số ảnh validation:", len(image_files))
print("Số file nhãn:", len(label_files))

assert BEST_PT.exists(), f"Không tìm thấy weights: {BEST_PT}"
assert image_files, f"Không tìm thấy ảnh trong: {VAL_IMAGES}"
assert label_files, f"Không tìm thấy nhãn trong: {VAL_LABELS}"

## Bước 4 — Hàm tính IoU

Hàm này đo độ chồng lấp giữa hai box `[x1, y1, x2, y2]`. IoU càng gần 1 thì hai box càng khớp.

In [ ]:
def calculate_iou(box1, box2):
    # Phần giao nhau của hai box
    intersection_x1 = max(box1[0], box2[0])
    intersection_y1 = max(box1[1], box2[1])
    intersection_x2 = min(box1[2], box2[2])
    intersection_y2 = min(box1[3], box2[3])

    intersection_width = max(0, intersection_x2 - intersection_x1)
    intersection_height = max(0, intersection_y2 - intersection_y1)
    intersection_area = intersection_width * intersection_height

    # Diện tích từng box và phần hợp
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / (union_area + 1e-9)

## Bước 5 — Hàm chia vật thể theo kích thước

Ảnh gốc có thể có nhiều kích thước khác nhau. Hàm scale chiều rộng và chiều cao box theo cách YOLO đưa ảnh về `640`, rồi dùng các ngưỡng `32²` và `96²`.

In [ ]:
def get_size_group(box_width, box_height, image_width, image_height):
    # Tỉ lệ resize ảnh trước khi đưa vào model
    scale = min(IMGSZ / image_width, IMGSZ / image_height)

    # Kích thước box mà model nhìn thấy ở đầu vào
    resized_width = box_width * scale
    resized_height = box_height * scale
    resized_area = resized_width * resized_height

    if resized_area < 32 ** 2:
        return "small"
    elif resized_area < 96 ** 2:
        return "medium"
    else:
        return "large"

## Bước 6 — Hàm đọc ground truth

Nhãn YOLO có dạng `class x_center y_center width height`, trong đó tọa độ được chuẩn hóa từ 0 đến 1. Hàm đổi chúng sang tọa độ pixel của ảnh gốc và xác định size của mỗi vật thể.

In [ ]:
def load_ground_truths(label_path, image_width, image_height):
    ground_truths = []

    if not label_path.exists():
        return ground_truths

    with open(label_path, "r", encoding="utf-8") as file:
        for line in file:
            values = line.strip().split()
            if len(values) != 5:
                continue

            class_id, x_center, y_center, box_width, box_height = map(float, values)

            # Đổi tọa độ YOLO 0-1 sang pixel ảnh gốc
            x_center *= image_width
            y_center *= image_height
            box_width *= image_width
            box_height *= image_height

            x1 = x_center - box_width / 2
            y1 = y_center - box_height / 2
            x2 = x_center + box_width / 2
            y2 = y_center + box_height / 2

            ground_truths.append({
                "class_id": int(class_id),
                "box": np.array([x1, y1, x2, y2]),
                "size": get_size_group(
                    box_width, box_height, image_width, image_height
                )
            })

    return ground_truths

## Bước 7 — Load model và predict tập validation

YOLO tự resize ảnh về `640` để dự đoán, sau đó tự đưa `result.boxes.xyxy` trở lại tọa độ ảnh gốc. Vì vậy prediction có thể so sánh trực tiếp với ground truth ở trên.

In [ ]:
model = YOLO(str(BEST_PT))

prediction_results = model.predict(
    source=str(VAL_IMAGES),
    imgsz=IMGSZ,
    conf=CONFIDENCE,
    device=0,
    stream=True,
    verbose=False
)

## Bước 8 — Ghép prediction với ground truth

Prediction được xét theo confidence từ cao xuống thấp. Mỗi prediction chỉ được ghép với một ground truth cùng class, và mỗi ground truth chỉ được ghép một lần.

In [ ]:
stats = {
    "small": {"ground_truth": 0, "detected": 0},
    "medium": {"ground_truth": 0, "detected": 0},
    "large": {"ground_truth": 0, "detected": 0}
}

for image_number, result in enumerate(prediction_results, start=1):
    image_path = Path(result.path)
    image_height, image_width = result.orig_shape
    label_path = VAL_LABELS / f"{image_path.stem}.txt"

    # Ground truth của ảnh hiện tại
    ground_truths = load_ground_truths(label_path, image_width, image_height)

    # Đếm tổng số ground truth theo size
    for ground_truth in ground_truths:
        stats[ground_truth["size"]]["ground_truth"] += 1

    if result.boxes is None or len(result.boxes) == 0:
        continue

    predicted_boxes = result.boxes.xyxy.cpu().numpy()
    predicted_classes = result.boxes.cls.cpu().numpy().astype(int)
    predicted_confidences = result.boxes.conf.cpu().numpy()

    # Prediction có confidence cao được xét trước
    prediction_order = np.argsort(-predicted_confidences)
    matched_ground_truths = set()

    for prediction_index in prediction_order:
        predicted_box = predicted_boxes[prediction_index]
        predicted_class = predicted_classes[prediction_index]

        # Chỉ lấy ground truth cùng class và chưa được ghép
        candidate_indices = [
            index for index, ground_truth in enumerate(ground_truths)
            if index not in matched_ground_truths
            and ground_truth["class_id"] == predicted_class
        ]

        if not candidate_indices:
            continue

        # Tìm ground truth có IoU lớn nhất với prediction hiện tại
        ious = [
            calculate_iou(predicted_box, ground_truths[index]["box"])
            for index in candidate_indices
        ]
        best_position = int(np.argmax(ious))
        best_iou = ious[best_position]

        if best_iou >= IOU_THRESHOLD:
            matched_index = candidate_indices[best_position]
            matched_ground_truths.add(matched_index)
            matched_size = ground_truths[matched_index]["size"]
            stats[matched_size]["detected"] += 1

    if image_number % 100 == 0:
        print(f"Đã xử lý {image_number}/{len(image_files)} ảnh")

## Bước 9 — Tính Recall

`detected` là số ground truth được ghép đúng (TP). `missed` là số ground truth không được ghép (FN). Prediction dư không ảnh hưởng Recall mà ảnh hưởng Precision.

In [ ]:
print("=== RECALL THEO KÍCH THƯỚC ===")

for size in ["small", "medium", "large"]:
    ground_truth_count = stats[size]["ground_truth"]
    detected_count = stats[size]["detected"]
    missed_count = ground_truth_count - detected_count
    recall = detected_count / ground_truth_count if ground_truth_count > 0 else 0

    print(
        f"{size.upper():7s} | "
        f"GT: {ground_truth_count:6d} | "
        f"Detected: {detected_count:6d} | "
        f"Missed: {missed_count:6d} | "
        f"Recall: {recall * 100:.2f}%"
    )

## Cách đọc kết quả

Ví dụ nếu kết quả là `small=20%`, `medium=55%`, `large=80%`, model bỏ sót vật thể nhỏ nhiều hơn rõ rệt. Khi so sánh với giải pháp mới, phải giữ nguyên `IMGSZ`, `CONFIDENCE`, `IOU_THRESHOLD` và tập validation.